In [2]:
import streamlit as st
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [3]:
# -----------------------------
# Load Data
# -----------------------------
meta_df = pd.read_excel("src/chinmay/Sample Meta.xlsx")
reviews_df = pd.read_excel("src/chinmay/Sample Reviews.xlsx")

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cpu"   # VERY IMPORTANT
)

print("Model loaded successfully")

/opt/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


: 

In [4]:
review_texts = reviews_df["text"].fillna("").tolist()
review_embeddings = embedder.encode(review_texts, convert_to_numpy=True)

index = faiss.IndexFlatL2(review_embeddings.shape[1])
index.add(review_embeddings)

In [5]:
# -----------------------------
# Summarizer (LLM)
# -----------------------------
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")


In [6]:
# -----------------------------
# Streamlit UI
# -----------------------------
st.title("📝 Amazon Product Review Summarizer (RAG-based)")


2026-03-01 17:21:52.174 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 17:21:52.383 
  command:

    streamlit run /opt/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-03-01 17:21:52.383 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 17:21:52.384 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [7]:
# Dropdown for product titles
product_options = meta_df["title"].dropna().unique().tolist()
selected_product = st.selectbox("Select a Product Title:", product_options)


2026-03-01 17:22:05.003 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 17:22:05.004 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 17:22:05.005 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 17:22:05.005 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 17:22:05.005 Session state does not function when running a script without `streamlit run`
2026-03-01 17:22:05.005 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 17:22:05.006 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 17:22:05.006 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [9]:
# Submit button
if st.button("Generate Summary"):
    # Step 1: Find product metadata
    product_row = meta_df[meta_df["title"] == selected_product]
    
    if not product_row.empty:
        asin = product_row.iloc[0]["parent_asin"]
        '''
        st.subheader("📦 Product Details")
        st.write(f"**Title:** {product_row.iloc[0]['title']}")
        st.write(f"**Average Rating:** {product_row.iloc[0]['average_rating']}")
        st.write(f"**Rating Count:** {product_row.iloc[0]['rating_number']}")
        st.write(f"**Features:** {product_row.iloc[0]['features']}")
        '''
        # Display product images if available
        if isinstance(product_row.iloc[0]["images"], list) and len(product_row.iloc[0]["images"]) > 0:
            st.subheader("🖼 Product Images")
            for img in product_row.iloc[0]["images"]:
                st.image(img["large"], width=200)
        
        # Step 2: Get reviews for this product
        product_reviews = reviews_df[reviews_df["parent_asin"] == asin]["text"].fillna("").tolist()
        
        if product_reviews:
            # Step 3: Retrieve top reviews using FAISS
            query_embedding = embedder.encode([selected_product], convert_to_numpy=True)
            D, I = index.search(query_embedding, k=5)
            top_reviews = [review_texts[i] for i in I[0] if i < len(review_texts)]
            
            # Step 4: Summarize reviews
            joined_reviews = " ".join(top_reviews)
            summary = summarizer(joined_reviews, max_length=150, min_length=50, do_sample=False)[0]["summary_text"]
            
            st.subheader("📝 Review Summary")
            st.write(summary)
        else:
            st.warning("No reviews found for this product.")
    else:
        st.error("Product not found in metadata.")

2026-03-01 17:23:19.351 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 17:23:19.351 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 17:23:19.352 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 17:23:19.352 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 17:23:19.353 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 17:23:19.353 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
